SystemMessage

HumanMessage

AIMessage

ToolMessage

##### josn格式

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
# 从.env文件中加载环境变量
load_dotenv(override=True)
BAILIAN_API_KEY = os.getenv("BAILIAN_API_KEY")
BAILIAN_API_URL = os.getenv("BAILIAN_API_URL")
model = init_chat_model(
model="qwen3.8-max",
model_provider="openai",
api_key=BAILIAN_API_KEY,
base_url=BAILIAN_API_URL
)
# 通过JSON初始化
messages = [
{"role": "system", "content": "你是一个善于给出通俗易懂解释的AI助手"},
{"role": "user", "content": "你好"},
{"role": "assistant", "content": "你好！我能帮你什么？"},
{"role": "user", "content": "什么是机器学习"}
]
response = model.invoke(messages)
print(response.content)

##### 消息队列

In [ ]:
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)
BAILIAN_API_KEY = os.getenv("BAILIAN_API_KEY")
BAILIAN_API_URL = os.getenv("BAILIAN_API_URL")
model = init_chat_model(
model="qwen3.8-max",
model_provider="openai",
api_key=BAILIAN_API_KEY,
base_url=BAILIAN_API_URL
)
# 通过JSON初始化
messages = [
SystemMessage("你是一个善于给出通俗易懂解释的AI助手"),
HumanMessage("你好"),
AIMessage("你好！我能帮你什么？"),
HumanMessage("什么是机器学习"),
]
response = model.invoke(messages)
print(response.content)

##### 工具调用(json 和 消息队列格式)

In [2]:
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage,ToolMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
# 从.env文件中加载环境变量
load_dotenv(override=True)
BAILIAN_API_KEY = os.getenv("BAILIAN_API_KEY")
BAILIAN_API_URL = os.getenv("BAILIAN_API_URL")
model = init_chat_model(
model="qwen3.8-max",
model_provider="openai",
api_key=BAILIAN_API_KEY,
base_url=BAILIAN_API_URL
)

def get_weather(city: str) -> str:
    return "不错哦~"

# 模拟模型绑定工具
model_with_tools = model.bind_tools([get_weather])
# ai_message = {
# "role": "assistant",
# "content": "",
# "tool_calls": [{
# "name": "get_weather",
# "args": {"location": "北京"},
# "id": "call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s"
# }]
# }
ai_message = AIMessage(
content = [],
tool_calls = [{
"name": "get_weather",
"args": {"location": "北京"},
"id": "call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s"
}]
)
# tool_message = {
# "role": "tool",
# "content": "今天北京天气晴朗，万里无云~",
# "tool_call_id": "call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s"
# }
tool_message = ToolMessage(
content = "今天北京天气晴朗，万里无云~",
tool_call_id = "call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s"
)
messages = [
# {"role": "user", "content": "北京天气如何"},
HumanMessage(content="北京天气如何"),
ai_message,
tool_message
]
# for message in messages:
# print(message)
response = model.invoke(messages)
print(response)


content='今天北京天气晴朗，万里无云，是个非常适合出门的好天气！☀️\n\n如果您有出行计划，记得做好防晒哦～还有什么其他想了解的吗？😊' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 116, 'total_tokens': 209, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 55, 'rejected_prediction_tokens': None, 'text_tokens': 93}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 116}}, 'model_provider': 'openai', 'model_name': 'qwen3.8-max', 'system_fingerprint': None, 'id': 'chatcmpl-e6d6ed93-e54d-97d3-988e-1d6038561b5d', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a13d6a-24ac-70d3-a0af-dba7556c8aa5-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 116, 'output_tokens': 93, 'total_tokens': 209, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 55}}


##### 历史对话优化

In [7]:
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage,ToolMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
# 从.env文件中加载环境变量
load_dotenv(override=True)
BAILIAN_API_KEY = os.getenv("BAILIAN_API_KEY")
BAILIAN_API_URL = os.getenv("BAILIAN_API_URL")
model = init_chat_model(
model="qwen3.8-max",
model_provider="openai",
api_key=BAILIAN_API_KEY,
base_url=BAILIAN_API_URL
)

def recent_messages(messages, max_pairs=3):
    """
    保留所有系统消息和最近 N 轮对话。
    适用于 HumanMessage、AIMessage 交替出现的完整对话。
    """
    if max_pairs < 0:
        raise ValueError("max_pairs 不能小于 0")

    # 分离系统消息和对话消息
    system_msgs = [
        m for m in messages if isinstance(m, SystemMessage)
    ]
    conversation_msgs = [
        m for m in messages if not isinstance(m, SystemMessage)
    ]

    # 每轮两条消息；0 表示不保留对话
    recent_msgs = (
        conversation_msgs[-(max_pairs * 2):]
        if max_pairs > 0 else []
    )

    return system_msgs + recent_msgs

In [ ]:
long_conversation = [
    SystemMessage("你是一个善于给出通俗易懂解释的数学老师")
]

long_conversation.append(HumanMessage("解释一下正弦定理"))
r1 = model.invoke(long_conversation)
long_conversation.append(AIMessage(r1.content))

long_conversation.append(HumanMessage("解释一下余弦定理"))
r2 = model.invoke(long_conversation)
long_conversation.append(AIMessage(r2.content))

long_conversation.append(HumanMessage("解释一下勾股定理"))  
r3 = model.invoke(long_conversation)
long_conversation.append(AIMessage(r3.content))

print(len(long_conversation))  # 输出消息总数
print(f"我第一个问的问题是{HumanMessage("我问第一个问题是什么")}")
print(f"{model.invoke(messages)}")
messages_to_send = recent_messages(long_conversation, max_pairs=3)

print(len(messages_to_send))  # 输出最近的消息对数
print(f"我第一个问的问题是{HumanMessage("我问第一个问题是什么")}")
print(f"{model.invoke(messages).content}")